# Day 020 — Exercise 3: retrieve

**Goal:** Implement `retrieve(query, collection, n_results)` that embeds the query and returns the n most similar chunks as `list[{text, source, distance}]`. **One real Ollama call** (for the query embedding) per check.

In [ ]:
import ollama
import chromadb

## Provided: chunk_text, embed_text, build_index

In [ ]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    words = text.split()
    step = chunk_size - overlap
    if step <= 0:
        step = 1
    chunks = []
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks


def embed_text(text: str, model: str = "nomic-embed-text") -> list[float]:
    return ollama.embeddings(model=model, prompt=text)["embedding"]


def build_index(
    docs: dict,
    collection_name: str = "second_brain",
    chunk_size: int = 300,
    overlap: int = 50,
):
    client = chromadb.Client()
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass
    collection = client.create_collection(collection_name)
    for source, text in docs.items():
        chunks = chunk_text(text, chunk_size, overlap)
        ids, embeddings, documents, metadatas = [], [], [], []
        for i, chunk in enumerate(chunks):
            ids.append(f"{source}__{i}")
            embeddings.append(embed_text(chunk))
            documents.append(chunk)
            metadatas.append({"source": source, "chunk_index": i})
        if ids:
            collection.add(
                ids=ids, embeddings=embeddings,
                documents=documents, metadatas=metadatas,
            )
    return collection

## Your Implementation

In [ ]:
def retrieve(query: str, collection, n_results: int = 3) -> list[dict]:
    """
    Semantic search. Returns list[{'text': str, 'source': str, 'distance': float}].
    Returns [] if the collection is empty.
    """
    # TODO: handle empty collection — return [] if collection.count() == 0
    # TODO: emb = embed_text(query)
    # TODO: actual_n = min(n_results, collection.count())
    # TODO: results = collection.query(query_embeddings=[emb], n_results=actual_n)
    # TODO: zip results['documents'][0], results['metadatas'][0], results['distances'][0]
    # TODO: return list of dicts with keys 'text', 'source', 'distance'
    pass

## Check Your Work

In [ ]:
TEST_DOCS_3 = {
    "python.txt": "Python is a high-level programming language with clean syntax.",
    "history.txt": "Artificial intelligence research began in the 1950s with Alan Turing.",
}

def _run_checks():
    total = 5
    passed = 0
    col = build_index(TEST_DOCS_3, collection_name='test_retrieve_020', chunk_size=20, overlap=3)
    results = None

    # Check 1: defined
    try:
        assert 'retrieve' in globals()
        passed += 1; print('✅ Check 1: retrieve defined')
    except Exception as e:
        print(f'❌ Check 1: {e}')

    # Check 2: returns a list (1 embed call)
    try:
        results = retrieve('python programming', col, n_results=2)
        assert isinstance(results, list), f'expected list, got {type(results)}'
        passed += 1; print('✅ Check 2: retrieve returns a list')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: items have required keys
    try:
        assert results is not None and len(results) > 0, 'results list is empty'
        for r in results:
            for key in ('text', 'source', 'distance'):
                assert key in r, f"missing key '{key}' in result: {r}"
        passed += 1; print('✅ Check 3: each result has text, source, distance')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: n_results limits output (1 embed call)
    try:
        r1 = retrieve('language', col, n_results=1)
        assert len(r1) <= 1, f'expected <= 1 result, got {len(r1)}'
        passed += 1; print('✅ Check 4: n_results limits the number of results')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: empty collection returns []
    try:
        _client = chromadb.Client()
        try: _client.delete_collection('empty020col')
        except: pass
        _empty = _client.create_collection('empty020col')
        empty_res = retrieve('anything', _empty, n_results=3)
        assert empty_res == [], f'expected [] for empty collection, got {empty_res}'
        passed += 1; print('✅ Check 5: empty collection returns []')
    except Exception as e:
        print(f'❌ Check 5: empty collection — {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')

_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def retrieve(query: str, collection, n_results: int = 3) -> list[dict]:
    if collection.count() == 0:
        return []
    emb = embed_text(query)
    actual_n = min(n_results, collection.count())
    results = collection.query(query_embeddings=[emb], n_results=actual_n)
    return [
        {"text": doc, "source": meta["source"], "distance": dist}
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]
```

</details>